# 02 — Layer Pruning (TrimLLM)

Drop the last ~25% of transformer layers, recover with QLoRA + LISA.

**Papers**
- TrimLLM https://arxiv.org/abs/2412.11242
- Reassessing Layer Pruning https://arxiv.org/abs/2411.15558
- LISA https://arxiv.org/abs/2403.17919

In [ ]:
# ###### Colab bootstrap ######
# On Colab the [experiments] extras pulls both requirements_package.txt and
# requirements_experiments.txt in one pip command — single source of truth lives
# in setup.py's extras_require. Then bootstrap() loads Colab Secrets into
# os.environ and brings up Tailscale so *.ts.net hostnames are reachable.
# Locally bootstrap() just loads .env and checks the tailnet — no installs.
#
# Required Colab Secrets (key icon → Add new secret → toggle "Notebook access"):
#   TAILSCALE_AUTHKEY   – from https://login.tailscale.com/admin/settings/keys
#   MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_INSECURE_TLS
#   MINIO_ENDPOINT / MINIO_ACCESS_KEY / MINIO_SECRET_KEY / MINIO_SECURE
#   HF_TOKEN
import os, subprocess, sys
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "burnit_bg[experiments] @ git+https://github.com/kirilyotov/BurnIT-BG.git",
    ])

from utils.colab import bootstrap
bootstrap()


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {props.name}")
    print(f"VRAM:   {props.total_memory / 1024**3:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")


## 1. Setup & config

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'data_platform').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from data_platform.common import set_env
from data_platform.storage import MinioStorage

from experiments.shared.mlflow_utils import setup_run, log_responses, stage, log_training_curves
from experiments.shared.inference_utils import (
    run_full_test_battery,
    TEST_PROMPTS_IN_DOMAIN, TEST_PROMPTS_OUT_OF_DOMAIN, TEST_PROMPTS_EDGE,
    format_prompt,
)
from experiments.shared.model_utils import (
    DEFAULT_MODEL_NAME, load_model_unsloth, apply_qlora, count_trainable_params,
    cuda_device_info,
)
from experiments.shared.eval_utils import compute_perplexity, benchmark_speed, vram_snapshot
from experiments.shared.dataset_utils import (
    load_alpaca_dataset, dataset_statistics, alpaca_to_prompt,
)
from experiments.shared.dataset_browser import list_datasets, pick_dataset, download_dataset, resolve


In [ ]:
set_env(quiet=True)
tracking, tags, run_name = setup_run(
    experiment='layer_pruning',
    model=DEFAULT_MODEL_NAME,
    stage="experiment",
    mlflow_experiment_name='burnit-bg-experiments',
)
print(f"run_name = {run_name}")
print(f"tags     = {tags}")
print(f"machine  = {cuda_device_info()}")


## 2. Data loading

In [ ]:
# Set DATASET_PREFIX in .env / Colab secrets to skip the picker, e.g.
#   DATASET_PREFIX=data_prep/processed/mental-health
# Or pass `auto=True` below for non-interactive runs.

DEFAULT_PREFIX = os.getenv("DATASET_PREFIX", "data_prep/processed")
try:
    local_dataset_dir = resolve(prefix=DEFAULT_PREFIX, auto=False)
    TRAIN_PATH = local_dataset_dir / "train.jsonl"
    EVAL_PATH  = local_dataset_dir / "eval.jsonl"
except (FileNotFoundError, Exception) as exc:
    print(f"[data] MinIO lookup failed ({exc}); falling back to local data_prep/processed/")
    TRAIN_PATH = REPO_ROOT / "data_prep/processed/train.jsonl"
    EVAL_PATH  = REPO_ROOT / "data_prep/processed/eval.jsonl"

train_records = list(load_alpaca_dataset(TRAIN_PATH))
eval_records  = list(load_alpaca_dataset(EVAL_PATH))
train_stats = dataset_statistics(train_records)
eval_stats  = dataset_statistics(eval_records)
print(f"train: {len(train_records)} rows  eval: {len(eval_records)} rows")
print(train_stats)


## 2b. Load the baseline + measure layer similarity

In [ ]:
# Load baseline weights from MLflow (replace with your baseline run-id).
# Falls back to the bare base model if no baseline run is given.
BASELINE_RUN_ID = os.getenv("BASELINE_RUN_ID")  # set in env or hardcode
model, tokenizer = load_model_unsloth(DEFAULT_MODEL_NAME, max_seq_length=2048,
                                      load_in_4bit=True, token=os.getenv("HF_TOKEN"))
if BASELINE_RUN_ID:
    try:
        local_path = tracking.load_model(run_id=BASELINE_RUN_ID, artifact_path="model")
        print(f"loaded baseline weights from {local_path}")
    except Exception as exc:
        print(f"[warn] baseline load failed ({exc}); continuing with bare base model")

# Layer similarity: capture hidden states with forward hooks, compute
# cosine similarity between consecutive layers.
import torch
from torch.nn.functional import cosine_similarity

@torch.no_grad()
def measure_layer_similarity(model, tokenizer, samples, max_tokens=256):
    layers = model.model.layers if hasattr(model, "model") else model.layers
    hidden_per_layer = []
    handles = []
    def make_hook(idx):
        def _hook(_module, _input, output):
            h = output[0] if isinstance(output, tuple) else output
            hidden_per_layer.append((idx, h.mean(dim=1).detach().cpu()))
        return _hook
    for i, layer in enumerate(layers):
        handles.append(layer.register_forward_hook(make_hook(i)))
    try:
        for text in samples:
            ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_tokens).to(model.device)
            model(**ids)
    finally:
        for h in handles: h.remove()
    # Aggregate per layer
    by_layer = {}
    for i, h in hidden_per_layer:
        by_layer.setdefault(i, []).append(h)
    means = [torch.cat(by_layer[i]).mean(dim=0) for i in sorted(by_layer)]
    sims = []
    for i in range(len(means) - 1):
        a, b = means[i], means[i+1]
        sims.append(float(cosine_similarity(a, b, dim=0).item()))
    return sims

similarities = measure_layer_similarity(
    model, tokenizer,
    samples=[alpaca_to_prompt(r) for r in eval_records[:32]],
)
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3.5))
plt.plot(range(len(similarities)), similarities, marker="o")
plt.axhline(0.95, color="crimson", linestyle="--", label="prune-candidate threshold")
plt.xlabel("Layer index"); plt.ylabel("cos(layer_i, layer_{i+1})")
plt.title("Layer-to-layer similarity"); plt.legend(); plt.tight_layout()
plt.savefig("/tmp/layer_sim.png", dpi=140)
tracking.log_metrics({f"layer_sim.{i}": s for i, s in enumerate(similarities)})
plt.show()


## 3. Prune the last 25% of layers

In [ ]:
# Remove the trailing ~25% of decoder blocks.
total_layers = len(model.model.layers)
prune_from = int(round(total_layers * 0.75))
keep = model.model.layers[:prune_from]
removed = total_layers - len(keep)
print(f"pruning layers {prune_from}..{total_layers - 1} ({removed} layers)")

import torch.nn as nn
model.model.layers = nn.ModuleList(keep)
if hasattr(model.config, "num_hidden_layers"):
    model.config.num_hidden_layers = len(keep)

params_after = count_trainable_params(model)
tracking.log_params({
    "prune.total_layers_before": total_layers,
    "prune.total_layers_after": len(keep),
    "prune.removed_layers": removed,
    "prune.params_after": params_after["total"],
})


## 4. Recovery fine-tuning (QLoRA + LISA)

In [ ]:
# Re-apply QLoRA to the pruned model so we can recover quickly.
model = apply_qlora(model, r=16, lora_alpha=32)

# LISA: at each step, freeze all decoder blocks except a randomly-chosen K.
# This dramatically lowers VRAM and trains a different subset every step.
import random, torch
ACTIVE_LAYERS_PER_STEP = 4

def lisa_select(model, k=ACTIVE_LAYERS_PER_STEP):
    layers = model.model.layers
    chosen = set(random.sample(range(len(layers)), k=min(k, len(layers))))
    for i, layer in enumerate(layers):
        for p in layer.parameters():
            p.requires_grad_(i in chosen)
    return chosen

from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback

train_ds = Dataset.from_list(train_records).map(lambda r: {**r, "text": alpaca_to_prompt(r, eos_token=tokenizer.eos_token)})
eval_ds  = Dataset.from_list(eval_records).map(lambda r: {**r, "text": alpaca_to_prompt(r, eos_token=tokenizer.eos_token)})

class LISACallback(TrainerCallback):
    def on_step_begin(self, args, state, control, **kw):
        lisa_select(kw["model"])

OUTPUT_DIR = REPO_ROOT / "tmp/experiments/layer_pruning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), num_train_epochs=3,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=1e-4, warmup_ratio=0.03, logging_steps=10,
    save_strategy="epoch", bf16=True, optim="adamw_8bit",
    report_to=[], lr_scheduler_type="cosine", max_grad_norm=1.0,
)

with tracking.run(run_name=run_name, tags=tags):
    tracking.log_params({"lisa.active_per_step": ACTIVE_LAYERS_PER_STEP})
    with stage(tracking, "recovery_train"):
        trainer = SFTTrainer(
            model=model, tokenizer=tokenizer,
            train_dataset=train_ds, eval_dataset=eval_ds,
            dataset_text_field="text", max_seq_length=2048,
            args=training_args, callbacks=[LISACallback()],
        )
        trainer.train()
        steps = [h["step"] for h in trainer.state.log_history if "loss" in h]
        losses = [h["loss"]  for h in trainer.state.log_history if "loss" in h]
        if steps:
            log_training_curves(tracking, steps=steps, losses=losses, title="recovery")


## 5. Evaluation — perplexity & speed comparison

In [ ]:
with stage(tracking, "evaluate"):
    ppl = compute_perplexity(model, tokenizer, [r["output"] for r in eval_records[:64]])
    speed = benchmark_speed(model, tokenizer, new_tokens=128, runs=3)
    tracking.log_metrics({
        "eval_perplexity": float(ppl),
        "tokens_per_sec": speed["tokens_per_sec_mean"],
        **{f"vram.{k}": v for k, v in vram_snapshot().items()},
    })
    print(f"perplexity={ppl:.3f}  tokens/sec={speed['tokens_per_sec_mean']:.1f}")


## 6. Inference test

In [ ]:
# ###### Inference test (mental-health prompts) ######
with stage(tracking, "inference_test"):
    batteries = run_full_test_battery((model, tokenizer), max_new_tokens=256)
    log_responses(
        tracking,
        experiment='layer_pruning',
        model_checkpoint=str(REPO_ROOT / "tmp/experiments/layer_pruning"),
        **batteries,
    )
    for k, v in batteries.items():
        print(f"-- {k} --")
        for entry in v[:2]:
            print(f"  Q: {entry['prompt'][:80]}\n  A: {entry['response'][:200]}\n")


## 7. Save via MLflow + GGUF export

In [ ]:
#  Save model via MLflow
# tracking.log_model logs the model artifact under runs:model and,
# when registered_model_name is set, adds a new version to the Models tab.
# MLflow's artifact store backs onto MinIO — no separate upload needed.
with stage(tracking, "save_model"):
    try:
        tracking.log_model(
            model,
            flavor="transformers",
            artifact_path="model",
            registered_model_name='burnit-bg-layer-pruning',
            input_example=None,
        )
        print("[save] model logged via MLflow")
    except Exception as exc:
        print(f"[save] log_model failed: {exc}")

# Optional: GGUF export for offline local inference (RTX 3050 / Ollama).
# The GGUF is added as a run artifact under `gguf/`.
with stage(tracking, "gguf_export"):
    try:
        from experiments.shared.model_utils import export_gguf
        gguf_path = export_gguf(model, tokenizer, REPO_ROOT / "tmp/experiments/layer_pruning/gguf", quantization="q4_k_m")
        tracking.save_data(gguf_path, artifact_path="gguf")
        print(f"[save] GGUF logged: {gguf_path}")
    except Exception as exc:
        print(f"[save] GGUF export skipped: {exc}")
